In [13]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Database connection parameters
db_params = {
    'dbname': 'dbt_equipment_losses',
    'user': os.getenv('DBT_USER'),
    'password': os.getenv('DBT_PASS'),
    'host': 'localhost',
    'port': 5432
}

# Establish a connection to the database
conn = psycopg2.connect(**db_params)

# Create a cursor object
cur = conn.cursor()

# Execute a SELECT query
cur.execute("SELECT * FROM public.cumulative_losses ORDER BY predicted_category, date_recorded")

# Fetch all rows from the result
rows = cur.fetchall()

# Get column names
column_names = [desc[0] for desc in cur.description]

# Create a pandas DataFrame
df = pd.DataFrame(rows, columns=column_names)

# Close the cursor and connection
cur.close()
conn.close()

# Display the first few rows of the DataFrame
#print(df.head())

# You can now work with the DataFrame 'df' which contains your data
# For example, you can perform analysis, create visualizations, etc.

# To save the DataFrame to a CSV file:
# df.to_csv('equipment_analysis.csv', index=False)

# 1. Convert date_recorded to datetime
df['date_recorded'] = pd.to_datetime(df['date_recorded'])

# 2. Ensure cumulative_loss_count is numeric
df['cumulative_loss_count'] = pd.to_numeric(df['cumulative_loss_count'], errors='coerce')

# Fill NaN values in cumulative_loss_count with 0 or forward fill
df['cumulative_loss_count'] = df['cumulative_loss_count'].fillna(method='ffill').fillna(0)

# For other columns, drop rows with NaN values if any
df = df.dropna()

# 4. Check for and remove any duplicate rows
df = df.drop_duplicates()


/var/folders/dv/ts7drh5n0hb48yx2_5vpm3kw0000gq/T/ipykernel_1707/4083927756.py:56: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['cumulative_loss_count'] = df['cumulative_loss_count'].fillna(method='ffill').fillna(0)


In [14]:
import pandas as pd
import hvplot.pandas
import holoviews as hv
from holoviews import render
import bokeh.io

# Assuming you've already loaded your data into a DataFrame called 'df'
# If not, you can load it like this:
# df = pd.read_sql("SELECT * FROM public.cumulative_losses ORDER BY predicted_category, country, date_recorded", your_database_connection)

# Create a function to generate a plot for a single category
def plot_category(category_df, category_name):
    return category_df.hvplot.line(
        x='date_recorded',
        y='cumulative_loss_count',
        by='country',
        title=f'Cumulative Losses for {category_name}',
        xlabel='Date',
        ylabel='Cumulative Loss Count',
        width=800,
        height=400,
        legend='right'
    )

# Create a list to store plots for each category
category_plots = []

# Generate plots for each category
for category in df['predicted_category'].unique():
    category_df = df[df['predicted_category'] == category]
    category_plots.append(plot_category(category_df, category))

# Combine all plots into a single layout
combined_plot = hv.Layout(category_plots).cols(2)

# Display the combined plot
# combined_plot

# Save as interactive HTML
bokeh.io.output_file("cumulative_losses.html")
bokeh.io.save(hv.render(combined_plot))


'/Users/culley/Documents/clones/RussiaUkraineWarEquipmentLosses/dbt/cumulative_losses.html'